In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
import kagglehub
from pathlib import Path

# Shoe vs Sandal vs Boot (15k images), downloaded once to the kagglehub cache
SHOE_DIR = Path(kagglehub.dataset_download("hasibalmuzdadid/shoe-vs-sandal-vs-boot-dataset-15k-images")) / "Shoe vs Sandal vs Boot Dataset"

C:\Users\jorda\Code\ml-notebooks\courses\deep-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0.00/47.0M [00:00<?, ?B/s]

  2%|▏         | 1.00M/47.0M [00:00<00:39, 1.24MB/s]

  4%|▍         | 2.00M/47.0M [00:01<00:20, 2.33MB/s]

  6%|▋         | 3.00M/47.0M [00:01<00:13, 3.48MB/s]

 11%|█         | 5.00M/47.0M [00:01<00:07, 6.13MB/s]

... (19 outputs left out)

 96%|█████████▌| 45.0M/47.0M [00:04<00:00, 15.9MB/s]

100%|██████████| 47.0M/47.0M [00:04<00:00, 16.0MB/s]

100%|██████████| 47.0M/47.0M [00:04<00:00, 11.8MB/s]

Extracting files...


In [3]:
image_size =32
path = str(SHOE_DIR)

In [4]:
images = []

for subdir_name in os.listdir(path):
    for image_name in os.listdir(f"{path}/{subdir_name}"):
        img = tf.keras.preprocessing.image.load_img(f"{path}/{subdir_name}/{image_name}", target_size =(image_size,image_size))
        images.append(img)
        
images = np.array(images)
images = np.float32(images)
images = images/255
        
images = np.reshape(images, (images.shape[0], image_size*image_size*3))


In [5]:
train_size = int(len(images)*0.6)
train_data = images [:train_size]
test_size = images [train_size:]



In [6]:
#model RBM

class RBM(object):
    def __init__(self, input_size, output_size, learning_rate=0.01, batch_size=128):
        self.input_size = input_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        
        self.w = tf.zeros([input_size, output_size], np.float32)
        self.hb = tf.zeros([output_size], np.float32)
        self.vb = tf.zeros([input_size], np.float32)
    
    def probability_h_given_v(self, visible, w, hb):
        return tf.nn.sigmoid(tf.matmul(visible, w)+ hb)
    
    def probability_v_given_h(self, hidden, w, vb):
        return tf.nn.sigmoid(tf.matmul(hidden, tf.transpose(w))+vb)
    
    def sample_probs(self,probs ):
        return tf.nn.relu(tf.sign(probs-tf.random.uniform(tf.shape(probs))))
    
    def update_weight_and_bias(self, positive_grad,negative_grad, batch,h0,h1,v1):
        self.w = self.w +self.learning_rate*(positive_grad-negative_grad)/ tf.dtypes.cast(tf.shape(batch)[0], tf.float32)
        self.vb = self.vb+self.learning_rate*tf.reduce_mean(batch - v1, 0)
        self.hb=self.hb+self.learning_rate*tf.reduce_mean(h0-h1,0)
        
    def train(self, x_train, epochs =10):
        losses=[]
        
        for epoch in range(epochs):
            for start, end in zip(range(0,len(x_train), self.batch_size), range (self.batch_size,len(x_train), self.batch_size)):
                batch = x_train[start:end]
                h0 = self.sample_probs(self.probability_h_given_v(batch,self.w, self.hb))
                v1 = self.sample_probs(self.probability_v_given_h(h0,self.w,self.vb))
                h1 = self.probability_h_given_v(v1, self.w, self.hb)
                
                positive_grad = tf.matmul(tf.transpose(batch),h0)
                negative_grad = tf.matmul(tf.transpose(v1),h1)
                self.update_weight_and_bias(positive_grad, negative_grad, batch, h0,h1,v1)
                
                loss = tf.reduce_mean(tf.square(batch-v1))
                losses.append(loss)
                
            return losses
        
    def RBM_reconstruct(self,x):
        h= self.probability_h_given_v(x, self.w, self.hb)
        
        reconstruct = self.probability_v_given_h(h,self.w,self.vb)
        
        
        
         

In [7]:
INPUT_SIZE = train_data.shape[1]
model = RBM(INPUT_SIZE,1000)
losses = model.train(train_data,10)